In [1]:
import json
with open('JAADall_results.json', 'r') as f:
    results = json.load(f)

In [2]:
def count_correlated_mistakes(preds_1, preds_2, gt):
    agreement = preds_1==preds_2
    stack_agreement_preds = []
    stack_gts = []

    for i, a in enumerate(agreement):
        if a==True:#two models agree each other
            stack_agreement_preds.append(int(preds_1[i]))
            stack_gts.append(int(gt[i]))
    aggre_num = len(np.array(stack_agreement_preds))
    mistakes_num = sum(np.array(stack_agreement_preds)!=np.array(stack_gts))

    return aggre_num, mistakes_num

def count_mistake_union(preds_1, preds_2, gt):
    mistake_1 = []#agreement
    mistake_2 = []#agreed mistakes
    
    for i, v in enumerate(gt):
        if int(gt[i])!= int(preds_1[i]):
            mistake_1.append(i)
        
        if int(gt[i])!= int(preds_2[i]):
            mistake_2.append(i)
    return len(set(list(set(mistake_1)) + list(set(mistake_2))))


def count_indivisual_mistakes(preds, gt):
    return sum(preds!=gt)

def get_conf_coef(probs_1, probs_2):
    return np.corrcoef(np.squeeze(probs_1), np.squeeze(probs_2))#x2_x3

# (1) x1+x2 vs x3

In [18]:
x1_x2_preds = np.array(results['x1_x2']['preds'])
x1_x2_probs = np.array(results['x1_x2']['probs'])
x1_x2_gts = np.array(results['x1_x2']['test_gts'])
x3_preds = np.array(results['x3']['preds'])
x3_probs = np.array(results['x3']['probs'])
x3_gts = np.array(results['x3']['test_gts'])

In [19]:
import numpy as np
preds_1 = x1_x2_preds
preds_2 = x3_preds
gt = x3_gts
count_correlated_mistakes(preds_1, preds_2, gt)

(315232, 39654)

In [20]:
count_mistake_union(preds_1, preds_2, gt)

195927

In [21]:
count_indivisual_mistakes(preds_1, gt)

84650

In [22]:
count_indivisual_mistakes(preds_2, gt)

150931

In [23]:
x1_x2_x3 = np.array(results['x1_x2_x3']['preds'])

In [24]:
preds_3 = x1_x2_x3
count_indivisual_mistakes(preds_3, gt)

56958

In [25]:
probs_1 = x1_x2_probs
probs_2 = x3_probs
get_conf_coef(probs_1, probs_2)

array([[1.        , 0.21802743],
       [0.21802743, 1.        ]])

# (2) x1+x3 vs x2

In [36]:
def get_all_correlation_info(id_1, id_2):
    print('testing ID_1={0} and ID_2={1} correlation'.format(id_1, id_2))
    id_1_preds = np.array(results[id_1]['preds'])
    id_1_probs = np.array(results[id_1]['probs'])
    id_1_gts = np.array(results[id_1]['test_gts'])
    id_2_preds = np.array(results[id_2]['preds'])
    id_2_probs = np.array(results[id_2]['probs'])
    id_2_gts = np.array(results[id_2]['test_gts'])
    
    if np.array_equal(id_1_gts, id_2_gts):
        print('agreements and correlate mistakes:')
        print(count_correlated_mistakes(id_1_preds, id_2_preds, id_2_gts))
        print('union mistakes: ')
        print(count_mistake_union(id_1_preds, id_2_preds, id_2_gts))
        print('individual mistake: ')
        print('ID_1: ')
        print(count_indivisual_mistakes(id_1_preds, gt))
        print('ID_2: ')
        print(count_indivisual_mistakes(id_2_preds, gt))
        print('confidence correlation: ')
        print(get_conf_coef(id_1_probs, id_2_probs))
    else:
        print('ID 1 and 2 should share the same ground truth!')
        sys.exit()

In [37]:
id_1 = 'x1_x3'
id_2 = 'x2'

get_all_correlation_info(id_1, id_2)

testing ID_1=x1_x3 and ID_2=x2 correlation
agreements and correlate mistakes:
(279441, 30321)
union mistakes: 
222385
individual mistake: 
ID_1: 
81749
ID_2: 
170957
confidence correlation: 
[[1.         0.05567891]
 [0.05567891 1.        ]]


In [39]:
id_1 = 'x2_x3'
id_2 = 'x1'

get_all_correlation_info(id_1, id_2)

testing ID_1=x2_x3 and ID_2=x1 correlation
agreements and correlate mistakes:
(334135, 36242)
union mistakes: 
173612
individual mistake: 
ID_1: 
118766
ID_2: 
91088
confidence correlation: 
[[1.         0.28323482]
 [0.28323482 1.        ]]
